# Rețele neuronale și deep learning — Seminarul 6: Bazele NLP: reprezentarea textului și reprezentarea vectorială a cuvintelor
Prof. dr. Daniel Traian Pele — Academia de Studii Economice din București (ASE); Academia Română, Institutul de Prognoză Economică (IPE)

[Deschide în Colab](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/main/notebooks/L6_seminar_ro.ipynb)

Sarcinile seminarului, pe datele reale din `../data/clean` (vezi `../data/SOURCES.md`). Scrieți codul în celulele goale de sub fiecare sarcină; păstrați valoarea fixă a generatorului aleator, ca rezultatele să poată fi comparate.

In [ ]:
# Google Colab: copiem repo-ul cursului, ca datele și rezultatele salvate să fie găsite
import os, subprocess, sys
if 'google.colab' in sys.modules and not os.getcwd().startswith('/content/NEURAL_BIZ'):
    if not os.path.isdir('/content/NEURAL_BIZ'):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/danpele/NEURAL_BIZ.git', '/content/NEURAL_BIZ'], check=True)
    os.chdir('/content/NEURAL_BIZ/notebooks')

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
DATA = '../data/clean/'
SEED = 20261108
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
except ImportError:
    DEVICE = 'cpu'
plt.rcParams.update({'figure.facecolor': 'none', 'axes.facecolor': 'none', 'axes.grid': False,
                     'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False})
print('device:', DEVICE)

## Pregătire

In [ ]:
import re
from collections import Counter
import torch, torch.nn as nn, torch.nn.functional as F
torch.set_num_threads(4)
TOK = re.compile(r"[a-z]+(?:'[a-z]+)?|\d+(?:[.,]\d+)*")
def tok(s):
    s = re.sub(r'https?://\S+|@\w+', ' ', str(s).lower())
    return ['<num>' if t[0].isdigit() else t for t in TOK.findall(s)]
pb = pd.read_csv(DATA + 'financial_phrasebank.csv')
tw = pd.read_csv(DATA + 'twitter_fin_sentiment.csv')
sp = pd.read_csv(DATA + 'ecb_speeches_2022plus.csv', parse_dates=['date'])
LAB = ['negative', 'neutral', 'positive']
pb_y = pb.label.map({k: i for i, k in enumerate(LAB)}).values
pb_tok = [tok(s) for s in pb.sentence]
print(len(pb), len(tw), len(sp))

In [ ]:
rs = np.random.RandomState(SEED); tr, va, te = [], [], []
for c in range(3):
    ids = rs.permutation(np.where(pb_y == c)[0]); n = len(ids)
    tr += list(ids[:int(.7 * n)]); va += list(ids[int(.7 * n):int(.85 * n)]); te += list(ids[int(.85 * n):])
tr, va, te = map(np.array, (tr, va, te))
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
J = lambda ids: [' '.join(pb_tok[i]) for i in ids]
def lr_tuned(Xtr, ytr, Xva, yva, Xte):
    best = max(((f1_score(yva, LogisticRegression(C=C, max_iter=3000, class_weight='balanced').fit(Xtr, ytr).predict(Xva), average='macro'), C)
                for C in [0.1, 1.0, 10.0]))
    m = LogisticRegression(C=best[1], max_iter=3000, class_weight='balanced').fit(Xtr, ytr)
    return m.predict(Xte), m
print(len(tr), len(va), len(te))

**Sarcina 1.** Calculați de mînă ponderea TF-IDF (tf sublinear, idf netezit) a unui cuvînt care apare de două ori într-o frază și în 30 din cele 3.000 de fraze de antrenare; verificați cu `TfidfVectorizer` pe un corpus mic.

In [ ]:
# codul vostru aici

**Sarcina 2.** Eliminați cuvintele de legătură din engleză (`stop_words='english'`) din modelul TF-IDF al cursului. Ce se întîmplă cu macro-F1 și ce cuvinte eliminate purtau sentiment?

In [ ]:
# codul vostru aici

**Sarcina 3.** Antrenați modelul TF-IDF + logistic numai pe frazele cu acord unanim al adnotatorilor și testați pe același set de test. Merită datele mai curate numărul mai mic de exemple?

In [ ]:
# codul vostru aici

**Sarcina 4.** Aplicați modelul de sentiment pentru tweet-uri (TF-IDF + logistic antrenat pe setul de antrenare al tweet-urilor) frazelor de test din PhraseBank, cu corespondența pesimist/neutru/optimist → negativ/neutru/pozitiv. Cît costă schimbarea domeniului?

In [ ]:
# codul vostru aici

**Sarcina 5.** Construiți un indice de sentiment al discursurilor BCE: aplicați modelul TF-IDF PhraseBank fiecărei fraze din fiecare discurs, mediați (pozitiv − negativ) pe trimestre și reprezentați grafic.

In [ ]:
# codul vostru aici